In [6]:
# Library imports
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from pathlib import Path
from collections import Counter
from collections import defaultdict
import os

# Visualization settings
plt.rcParams['figure.figsize'] = (12, 8)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# mostra o dataframe para caber certinho na tela
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 500)

In [7]:
# acessar a pasta experiments
dir_experiments="../experiments/prototypeEvaluation_filterReduction/results"

# Estrutura para armazenar os caminhos dos arquivos 
results_files = defaultdict(dict)

for folder in sorted(os.listdir(dir_experiments)):
    path = os.path.join(dir_experiments, folder)
    for file in sorted(os.listdir(path)):
       if file.endswith("_results.csv") or file.endswith("_layer3_test1-classified-images.cvs"):
                results_files[folder][file] = os.path.join(path, file)
           

In [8]:
results_data = []

for folder, contents in results_files.items():
    superpixel = int(folder.split("_")[0].replace('super', ''))
    nimage = int(folder.split("_")[1].replace('images', ''))
    technique = folder.split('_')[2]
    # print(f"{folder=}")
    # print(f"{superpixel=}, {nimage=}, {technique=}")
    for file, dir in contents.items():
        if isinstance(file, str) and file.endswith('_results.csv'):
            results_file = dir
            seed_value = int(file.replace('_results.csv', '').replace('seed', ''))
            with open(results_file, 'r') as f:
                lines = f.readlines()
                if len(lines) >= 4:
                    # Extract accuracy and kappa metrics from the first two lines
                    class1_accuracy, class2_accuracy = map(float, lines[0].strip().split(';')[:2])
                    kappa, global_accuracy = map(float, lines[1].strip().split(';')[:2])
                    # Try to extract nfeat from line 3 or 5 if available
                    if len(lines) > 5 and ': ' in lines[5]:
                        try:
                            nfeat = int(lines[5].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    elif len(lines) > 3 and ': ' in lines[3]:
                        try:
                            nfeat = int(lines[3].strip().split(': ')[1])
                        except (IndexError, ValueError):
                            nfeat = 0
                    # Append the extracted metrics to the results list
                    results_data.append({
                        'superpixel': superpixel,
                        'nimage': nimage,
                        'technique': technique,
                        'seed': seed_value,
                        'class1_accuracy': class1_accuracy,
                        'class2_accuracy': class2_accuracy,
                        'kappa': kappa,
                        'global_accuracy': global_accuracy,
                        'nfeat': nfeat
                    })


# Convert the results list to a DataFrame
df_technique = pd.DataFrame(results_data)

# Save the DataFrame to a CSV file for further analysis
# df_technique.to_csv('prototype_results_summary.csv', index=False)

In [9]:
df_technique.head(100)

,superpixel,nimage,technique,seed,class1_accuracy,class2_accuracy,kappa,global_accuracy,nfeat
0,150,2,cossine,1011,0.811659,0.978488,0.804012,0.957314,21160
1,150,2,cossine,1213,0.793722,0.976532,0.785302,0.953330,21160
2,150,2,cossine,123,0.816144,0.973273,0.789416,0.953330,21160
3,150,2,cossine,2735,0.789238,0.983051,0.804668,0.958452,21160
4,150,2,cossine,42,0.784753,0.982399,0.799317,0.957314,21160
...,...,...,...,...,...,...,...,...,...
75,150,5,euclidean,456,0.753363,0.977184,0.759660,0.948776,21160
76,150,5,euclidean,6854,0.780269,0.971317,0.758856,0.947069,21160
77,150,5,euclidean,7580,0.802691,0.975880,0.789156,0.953899,21160
78,150,5,euclidean,789,0.748879,0.973273,0.743522,0.944792,21160


In [10]:
# Define the metrics to be analyzed (excluding 'nfeat' for summary statistics)
metrics = ['class1_accuracy', 'class2_accuracy', 'kappa', 'global_accuracy', 'nfeat']

# Agrupa por superpixel, nimage e technique e calcula média e desvio padrão para cada métrica
summary_stats = df_technique.groupby(['superpixel', 'nimage', 'technique'])[metrics[:-1]].agg(['mean', 'std']).reset_index()

# Rename columns for easier access (e.g., 'kappa_mean', 'global_accuracy_mean')
summary_stats.columns = ['_'.join(col).strip() for col in summary_stats.columns.values]

# Arrange the DataFrame according to the order of superpixel values
# superpixels_values = sorted(superpixels_values)
# summary_stats = summary_stats.set_index('superpixel_').loc[superpixels_values].reset_index()

# Select only the metrics of interest for visualization and highlight the highest values
summary_stats = summary_stats[['superpixel_', 'nimage_', 'technique_', 'kappa_mean', 'kappa_std', 'global_accuracy_mean', 'global_accuracy_std']].style.highlight_max(
    subset=['kappa_mean', 'global_accuracy_mean'], color='gray'
)

# Format values as percentages for better presentation
summary_stats = summary_stats.format({'kappa_mean': '{:.4%}', 'global_accuracy_mean': '{:.4%}'})

# Display the styled DataFrame
summary_stats

,superpixel_,nimage_,technique_,kappa_mean,kappa_std,global_accuracy_mean,global_accuracy_std
0,150,2,cossine,80.2782%,0.028968,95.7428%,0.005677
1,150,2,euclidean,78.2771%,0.079336,95.1850%,0.019243
2,150,3,cossine,80.4826%,0.035525,95.7883%,0.007748
3,150,3,euclidean,77.6635%,0.078457,95.1053%,0.017622
4,150,4,cossine,79.3579%,0.055170,95.5435%,0.010925
5,150,4,euclidean,78.4154%,0.033626,95.3614%,0.006735
6,150,5,cossine,79.2448%,0.035182,95.5321%,0.007461
7,150,5,euclidean,79.6004%,0.038889,95.5663%,0.008354
